# **01_data_auditoria_y_preparación**

Este notebook realiza la auditoría, preparación y validación del dataset base
`energy_weather_holidays_daily_2022_2025.csv`.

## Objetivos
- Cargar y auditar el dataset base.
- Verificar consistencia temporal.
- Reindexar la serie a frecuencia diaria.
- Identificar y tratar días faltantes.
- Imputar la demanda faltante usando la mediana por día de la semana.
- Validar el dataset final.
- Exportar un archivo limpio y listo para análisis y modelado.

**BLOQUE 2 — Importación de librerías**

In [ ]:
import os
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

**BLOQUE 3 — Configuración de rutas**

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS_TABLES = PROJECT_ROOT / "outputs" / "tablas"
OUTPUTS_FIGURES = PROJECT_ROOT / "outputs" / "figuras"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUTS_TABLES.mkdir(parents=True, exist_ok=True)
OUTPUTS_FIGURES.mkdir(parents=True, exist_ok=True)

INPUT_FILE = DATA_RAW / "energy_weather_holidays_daily_2022_2025.csv"
OUTPUT_FILE = DATA_PROCESSED / "data_daily_ready.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Archivo de entrada:", INPUT_FILE)
print("Archivo de salida:", OUTPUT_FILE)

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo de entrada: {INPUT_FILE}\n"
        "Verifica que el dataset esté en data/raw/ dentro del repositorio."
    )

**BLOQUE 4 — Carga del dataset**

In [ ]:
df = pd.read_csv(INPUT_FILE)

print("Dimensión inicial:", df.shape)
display(df.head())
display(df.tail())

**BLOQUE 5 — Estandarización inicial**

In [ ]:
# Copia de seguridad
df = df.copy()

# Normalizar nombres de columnas
df.columns = [col.strip() for col in df.columns]

# Verificar columna de fecha
if "date" not in df.columns:
    raise ValueError("No se encontró la columna 'date' en el dataset.")

# Convertir fecha
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Revisar fechas inválidas
invalid_dates = df["date"].isna().sum()

print("Fechas inválidas:", invalid_dates)

if invalid_dates > 0:
    print("Filas con fecha inválida:")
    display(df[df["date"].isna()].head())
    raise ValueError("Existen fechas inválidas. Revisa el archivo antes de continuar.")

# Ordenar por fecha
df = df.sort_values("date").reset_index(drop=True)

print("Rango temporal inicial:")
print("Fecha mínima:", df["date"].min())
print("Fecha máxima:", df["date"].max())

**BLOQUE 6 — Auditoría inicial del dataset**

In [ ]:
print("Columnas del dataset:")
print(list(df.columns))

print("\nTipos de datos:")
display(df.dtypes.to_frame("dtype"))

print("\nValores nulos por columna:")
display(df.isna().sum().to_frame("null_count").sort_values("null_count", ascending=False))

print("\nDuplicados exactos:")
print(df.duplicated().sum())

print("\nDuplicados por fecha:")
duplicated_dates = df["date"].duplicated().sum()
print(duplicated_dates)

if duplicated_dates > 0:
    print("\nFechas duplicadas:")
    display(df[df["date"].duplicated(keep=False)].sort_values("date").head(20))

**BLOQUE 7 — Resolver duplicados por fecha**

In [ ]:
if duplicated_dates > 0:
    raise ValueError("Existen fechas duplicadas. Debes revisarlas antes de continuar.")
else:
    print("No existen fechas duplicadas por día. Se puede continuar.")

**BLOQUE 8 — Reindexación diaria**

In [ ]:
df = df.set_index("date").sort_index()

full_index = pd.date_range(start=df.index.min(), end=df.index.max(), freq="D")

expected_days = len(full_index)
observed_days = len(df)

print("Días esperados:", expected_days)
print("Días observados:", observed_days)
print("Huecos detectados:", expected_days - observed_days)

missing_dates = full_index.difference(df.index)

if len(missing_dates) > 0:
    missing_df = pd.DataFrame({"missing_date": missing_dates})
    print("\nPrimeras fechas faltantes:")
    display(missing_df.head(10))
else:
    print("No se detectaron huecos temporales.")

# Reindexar
df = df.reindex(full_index)
df.index.name = "date"

# Flag de observación faltante original
df["is_missing_original"] = df["demand_kwh_total"].isna().astype(int)

print("Dimensión tras reindexación:", df.shape)
display(df.head())

In [ ]:
print("Verificación del índice temporal:")

print("¿Índice ordenado?:", df.index.is_monotonic_increasing)
print("¿Frecuencia inferida?:", pd.infer_freq(df.index))
print("¿Duplicados en índice?:", df.index.duplicated().sum())

**BLOQUE 9 — Crear variables temporales auxiliares**

In [ ]:
df["year_check"] = df.index.year
df["month_check"] = df.index.month
df["dow_check"] = df.index.dayofweek

**BLOQUE 10 — Imputación de demanda**

In [ ]:
TARGET_COL = "demand_kwh_total"

if TARGET_COL not in df.columns:
    raise ValueError(f"No se encontró la columna objetivo '{TARGET_COL}'.")

# Mediana por día de semana calculada solo con datos observados
dow_medians = (
    df.loc[df[TARGET_COL].notna()]
      .groupby(df.loc[df[TARGET_COL].notna()].index.dayofweek)[TARGET_COL]
      .median()
)

print("Mediana de demanda por día de semana:")
display(dow_medians.to_frame("median_kwh"))

# Crear columna imputada
df["demand_imputed_kwh"] = df[TARGET_COL].copy()

missing_mask = df["demand_imputed_kwh"].isna()

df.loc[missing_mask, "demand_imputed_kwh"] = (
    df.loc[missing_mask].index.dayofweek.map(dow_medians)
)

# Flag de imputación
df["is_imputed_day"] = missing_mask.astype(int)

print("Total de días imputados:", int(df["is_imputed_day"].sum()))
print("Porcentaje imputado:", round(df["is_imputed_day"].mean() * 100, 2), "%")

**BLOQUE 11 — Validación de la imputación**

In [ ]:
print("Nulos en demanda original:", df[TARGET_COL].isna().sum())
print("Nulos en demanda imputada:", df["demand_imputed_kwh"].isna().sum())

if df["demand_imputed_kwh"].isna().sum() > 0:
    raise ValueError("La columna demand_imputed_kwh aún contiene nulos.")
else:
    print("Imputación completada correctamente.")

**BLOQUE 12 — Validación y reconstrucción ligera de calendario**

In [ ]:
# Variables calendario esperadas
calendar_cols_expected = [
    "year", "month", "dow", "weekofyear",
    "is_weekend", "is_business_day",
    "holiday_name", "is_holiday", "is_pre_holiday",
    "is_post_holiday", "is_long_weekend"
]

print("Validando columnas calendario...")
for col in calendar_cols_expected:
    print(f"{col}: {'OK' if col in df.columns else 'NO ENCONTRADA'}")

**BLOQUE 13 — Completar variables básicas si faltan**

In [ ]:
# year
if "year" not in df.columns:
    df["year"] = df.index.year

# month
if "month" not in df.columns:
    df["month"] = df.index.month

# dow
if "dow" not in df.columns:
    df["dow"] = df.index.dayofweek

# weekofyear
if "weekofyear" not in df.columns:
    df["weekofyear"] = df.index.isocalendar().week.astype(int)

# is_weekend
if "is_weekend" not in df.columns:
    df["is_weekend"] = df.index.dayofweek.isin([5, 6]).astype(int)

# is_business_day
if "is_business_day" not in df.columns:
    df["is_business_day"] = (~df.index.dayofweek.isin([5, 6])).astype(int)

# holiday_name
if "holiday_name" not in df.columns:
    df["holiday_name"] = ""

# Banderas de feriado
for col in ["is_holiday", "is_pre_holiday", "is_post_holiday", "is_long_weekend"]:
    if col not in df.columns:
        df[col] = 0

**BLOQUE 14 — Relleno conservador de columnas no objetivo**

In [ ]:
# Columnas binarias
binary_cols = ["is_weekend", "is_business_day", "is_holiday", "is_pre_holiday", "is_post_holiday", "is_long_weekend"]

for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0).astype(int)

# holiday_name
if "holiday_name" in df.columns:
    df["holiday_name"] = df["holiday_name"].fillna("")

# Variables temporales derivadas del índice
df["year"] = df.index.year
df["month"] = df.index.month
df["dow"] = df.index.dayofweek
df["weekofyear"] = df.index.isocalendar().week.astype(int)

**BLOQUE 15 — Tratamiento de meteorológicas**

In [ ]:
weather_cols = ["t_mean_c", "t_max_c", "t_min_c", "rh_mean_pct", "precip_mm", "sun_hours", "sst_c"]

existing_weather_cols = [col for col in weather_cols if col in df.columns]

print("Columnas meteorológicas detectadas:", existing_weather_cols)

if existing_weather_cols:
    print("\nNulos antes de interpolación:")
    display(df[existing_weather_cols].isna().sum().to_frame("null_count"))

    # Interpolación temporal
    df[existing_weather_cols] = df[existing_weather_cols].interpolate(method="time")

    # Relleno por arrastre si quedan bordes
    df[existing_weather_cols] = df[existing_weather_cols].ffill().bfill()

    print("\nNulos después de interpolación:")
    display(df[existing_weather_cols].isna().sum().to_frame("null_count"))

**BLOQUE 16 — Crear columna objetivo final oficial**

In [ ]:
df["target_kwh"] = df["demand_imputed_kwh"]

print("Nulos en target_kwh:", df["target_kwh"].isna().sum())

In [ ]:
print("Validación cruzada de variables clave:")

check_df = df[[
    "demand_kwh_total",
    "demand_imputed_kwh",
    "target_kwh",
    "is_imputed_day"
]].copy()

display(check_df.head(10))

print("\nChequeo coherencia:")

# Los días imputados deben coincidir con nulos originales
consistency_check = (
    (df["demand_kwh_total"].isna()) == (df["is_imputed_day"] == 1)
).all()

print("Consistencia imputación:", consistency_check)

**BLOQUE 17 — Validación final completa**

In [ ]:
final_checks = {
    "n_rows": len(df),
    "start_date": df.index.min(),
    "end_date": df.index.max(),
    "duplicate_index": int(df.index.duplicated().sum()),
    "missing_target_final": int(df["target_kwh"].isna().sum()),
    "imputed_days": int(df["is_imputed_day"].sum()),
    "imputed_pct": round(df["is_imputed_day"].mean() * 100, 2)
}

print("Resumen de validación final:")
for k, v in final_checks.items():
    print(f"{k}: {v}")

**BLOQUE 18 — Selección y orden final de columnas**

In [ ]:
preferred_order = [
    "demand_kwh_total",
    "demand_imputed_kwh",
    "target_kwh",
    "is_missing_original",
    "is_imputed_day",
    "year",
    "month",
    "weekofyear",
    "dow",
    "is_weekend",
    "is_business_day",
    "holiday_name",
    "is_holiday",
    "is_pre_holiday",
    "is_post_holiday",
    "is_long_weekend",
    "t_mean_c",
    "t_max_c",
    "t_min_c",
    "rh_mean_pct",
    "precip_mm",
    "sun_hours",
    "sst_c"
]

final_cols = [col for col in preferred_order if col in df.columns]
remaining_cols = [col for col in df.columns if col not in final_cols]

df_final = df[final_cols + remaining_cols].copy()

display(df_final.head())
print("Dimensión final:", df_final.shape)

**BLOQUE 19 — Exportación**

In [ ]:
df_final = df_final.reset_index().rename(columns={"date": "date"})
df_final.to_csv(OUTPUT_FILE, index=False)

print("Archivo exportado correctamente en:")
print(OUTPUT_FILE)

**BLOQUE 20 — Resumen final tipo TFM**

In [ ]:
summary_table = pd.DataFrame({
    "Indicador": [
        "Periodo analizado",
        "Registros esperados",
        "Registros finales",
        "Días imputados",
        "Porcentaje imputado"
    ],
    "Valor": [
        f"{df_final['date'].min()} a {df_final['date'].max()}",
        len(pd.date_range(df_final["date"].min(), df_final["date"].max(), freq="D")),
        len(df_final),
        int(df_final["is_imputed_day"].sum()),
        f"{round(df_final['is_imputed_day'].mean()*100, 2)} %"
    ]
})

display(summary_table)